# Predicting Smartphone Addiction — Final Solution

**Kaggle Playground Series S6E8** · metric **ROC-AUC** · final rank **717 / 3532**

| | |
|---|---|
| Best single model | XGBoost, 10-fold, OOF **0.968729** |
| Final blend | 14-model rank blend, OOF **0.969340** |
| Public LB | **0.97037** |
| Winner (reference) | 0.97176 private |

---

## The finding this solution is built on

The competition data is synthetic, generated from a real 7,500-row source dataset. The
generator **memorised a per-exact-value probability table** into the discrete lattice
columns. Measured standalone OOF AUC:

| column | as a float | as exact-value target encoding |
|---|---|---|
| `notifications_per_day` | **0.492** — nothing at all | **0.749** |
| `app_opens_per_day` | 0.541 | **0.735** |
| `sleep_hours` | 0.527 | 0.591 |
| `age` | 0.502 | 0.550 |

`notifications_per_day` has correlation −0.012 with the target and dead-flat deciles, so it
is invisible to every standard EDA view. Yet adjacent integers swing from 0.37 to 0.92 —
up to **129 binomial standard errors** apart. It is a lookup table, not a curve.

A raw-numeric LightGBM gives the game away: its top two features *by split count* are
`notifications_per_day` (9,373 splits) and `app_opens_per_day` (7,875), versus 2,512 for
`daily_screen_time_hours`, the strongest genuine predictor. The trees burn their entire
budget rebuilding the table by hand. **Out-of-fold target encoding hands it over directly:
+0.0033 AUC**, against +0.0010 for all conventional feature engineering combined.

## Contribution of each step

| step | OOF AUC | delta |
|---|---|---|
| raw numerics (12 features) | 0.963748 | — |
| + engineered features | 0.964772 | +0.00102 |
| **+ exact-value target encoding** | 0.967057 | **+0.00229** |
| + categorical TE + value counts | 0.967285 | +0.00023 |
| + engineered on top | 0.968335 | +0.00105 |
| + 10-fold, seed averaging, GPU | 0.968729 | +0.00039 |
| **+ 14-model rank blend** | **0.969340** | **+0.00061** |

> **Runtime.** The full pipeline is roughly 2 hours on an RTX 3050 (4 GB) plus 16 CPU
> cores. Set `QUICK = True` in the config cell for a 5-fold, single-seed smoke run.

## 1. Setup

In [ ]:
import os, sys, time, json, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

import lightgbm as lgb, xgboost as xgb, torch
from catboost import CatBoostClassifier
from concurrent.futures import ThreadPoolExecutor
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata

# ---------------- config ----------------
DATA   = os.environ.get("S6E8_DATA", "../data")   # folder holding train.csv / test.csv
OUT    = "../out"
QUICK  = False          # True -> 5 folds, 1 seed, fewer epochs (smoke run)
SEED   = 42
SMOOTH = 2.0            # target-encoding smoothing (tuned; 2 > 20 > 50 > 200)

N_SPLITS = 5 if QUICK else 10
N_SEEDS  = 1 if QUICK else 2
DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
os.makedirs(OUT, exist_ok=True)

TARGET = "addicted_label"
CAT = ["gender", "stress_level", "academic_work_impact"]
NUM = ["age", "daily_screen_time_hours", "social_media_hours", "gaming_hours",
       "work_study_hours", "sleep_hours", "notifications_per_day",
       "app_opens_per_day", "weekend_screen_time"]
FLOATC = ["daily_screen_time_hours", "social_media_hours", "gaming_hours",
          "work_study_hours", "sleep_hours", "weekend_screen_time"]
STEP = {"age": 1.0, "daily_screen_time_hours": .01, "social_media_hours": .01,
        "gaming_hours": .01, "work_study_hours": .01, "sleep_hours": .01,
        "notifications_per_day": 1.0, "app_opens_per_day": 1.0, "weekend_screen_time": .01}

print(f"folds={N_SPLITS} seeds={N_SEEDS} device={DEVICE} torch={torch.__version__}")

In [ ]:
train = pd.read_csv(f"{DATA}/train.csv")
test  = pd.read_csv(f"{DATA}/test.csv")
y = train[TARGET].values.astype("int64")
print(train.shape, test.shape, "| positive rate", round(y.mean(), 5))
train.head()

## 2. The evidence: per-value structure

Group by the **exact** value and compare each value's target rate against a rolling-smoothed
version of its neighbours. `z = (rate - smoothed) / binomial_se`. If the field were smooth,
`std(z)` would be about 1. Anything far above that is a memorised lookup table.

This single diagnostic is what the whole solution rests on — run it first on any synthetic
tabular competition.

In [ ]:
def lookup_zscore(df, col, target=TARGET, min_n=200, window=11):
    g = df[[col, target]].dropna().groupby(col)[target].agg(["mean", "size"])
    g = g[g["size"] >= min_n]
    smoothed = g["mean"].rolling(window, center=True, min_periods=3).mean()
    se = np.sqrt(g["mean"] * (1 - g["mean"]) / g["size"])
    z = ((g["mean"] - smoothed) / se).dropna()
    return len(g), z.std(), z.abs().max()

print(f"{'column':26s} {'n_values':>9s} {'std(z)':>8s} {'max|z|':>8s}   (std(z)~1 => smooth)")
for c in NUM:
    n, s, m = lookup_zscore(train, c)
    flag = "  <-- LOOKUP TABLE" if s > 5 else ""
    print(f"{c:26s} {n:9d} {s:8.2f} {m:8.1f}{flag}")

### The structural constraint

`daily_screen_time_hours >= social_media_hours + gaming_hours + work_study_hours` holds
**exactly** on every complete row — a hard invariant of the generator. The residual
`other_h` is the one hand-built feature that reliably pays.

In [ ]:
sub = train.dropna(subset=["daily_screen_time_hours", "social_media_hours",
                           "gaming_hours", "work_study_hours"])
resid = sub.daily_screen_time_hours - (sub.social_media_hours + sub.gaming_hours + sub.work_study_hours)
print(f"complete rows: {len(sub):,} | residual min = {resid.min():.4f} "
      f"| violations = {(resid < -1e-9).sum()}")

## 3. Feature engineering

In [ ]:
def _ndec(s):
    # Decimal-place count. MUST be read from the raw CSV text: pandas collapses
    # 7.0 and 7.00 into one float, and that distinction is real signal.
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return -1
    s = str(s)
    if s in ("nan", ""):
        return -1
    return len(s.split(".")[1].rstrip("0")) if "." in s else 0


def load_decimals(which):
    raw = pd.read_csv(f"{DATA}/{which}.csv", dtype=str, usecols=FLOATC)
    return pd.DataFrame({c: raw[c].map(_ndec).astype("int8") for c in FLOATC})


def build_features(df, dec):
    # Structural + decimal + behavioural + rule + weekend blocks.
    # Note: the widely-repeated advice to drop 'behavioural ratios' does NOT hold once
    # target encoding is present -- dropping them measured -0.00020 here.
    d = pd.DataFrame(index=df.index)
    dsh, smh = df.daily_screen_time_hours, df.social_media_hours
    gmh, wsh, wke = df.gaming_hours, df.work_study_hours, df.weekend_screen_time

    # structural -- generator invariants
    d["comp_sum"] = smh + gmh + wsh
    d["other_h"] = dsh - d.comp_sum
    d["nonsocial_h"] = dsh - gmh - wsh

    # decimal artifacts -- how hard the generator smeared each row
    for c in FLOATC:
        d[f"ndec_{c}"] = dec[c].to_numpy()
    w = (dec[FLOATC] == 0)
    d["n_whole"] = w.sum(axis=1).to_numpy().astype("float32")
    for k, c in [("dsh", "daily_screen_time_hours"), ("slp", "sleep_hours"), ("gam", "gaming_hours")]:
        d[f"whole_{k}"] = w[c].to_numpy().astype("float32")

    # behavioural
    d["social_share"] = smh / dsh
    d["gaming_share"] = gmh / dsh
    d["work_share"] = wsh / dsh
    d["notif_per_open"] = df.notifications_per_day / df.app_opens_per_day
    d["screen_per_open"] = dsh * 60 / df.app_opens_per_day
    d["awake_h"] = 24 - df.sleep_hours
    d["screen_share_awake"] = dsh / (24 - df.sleep_hours)

    # distance to the original source rule's thresholds
    d["d_minus_8"] = dsh - 8.0
    d["s_minus_4"] = smh - 4.0
    d["rule_margin"] = np.maximum(dsh - 8.0, (smh - 4.0) * 2.0)

    # weekend as a second noisy measurement of the same latent quantity
    d["wke_minus_dsh"] = wke - dsh
    d["wke_over_dsh"] = wke / dsh
    d["screen_mean"] = (dsh + wke) / 2
    d["screen_max"] = np.maximum(dsh, wke)

    for c in d.columns:
        d[c] = pd.to_numeric(d[c], errors="coerce").replace([np.inf, -np.inf], np.nan).astype("float32")
    return d


dec_tr, dec_te = load_decimals("train"), load_decimals("test")
feat_tr, feat_te = build_features(train, dec_tr), build_features(test, dec_te)
print("engineered features:", feat_tr.shape[1])

### Model-based imputation, then re-encode the recovered value

When `daily_screen_time_hours` is missing (13.9% of rows) its target encoding falls back to
the prior, so those rows lose the strongest feature outright. But it is recoverable —
`weekend_screen_time` correlates 0.80 with it, and the generator constraint bounds it from
below. Recover the value, clamp it to the constraint, snap it back to the 0.01 lattice, and
**target-encode the recovered value** so the per-value lookup exists for rows that had none.

The imputers never see the target, so they are fit once on train+test features pooled.

In [ ]:
IMPUTE_TARGETS = ["daily_screen_time_hours", "social_media_hours", "weekend_screen_time"]

def fit_impute(tr, te, cols_all, targets=IMPUTE_TARGETS, seed=0):
    pool = pd.concat([tr[cols_all], te[cols_all]], ignore_index=True)
    n_tr, out = len(tr), {}
    for t in targets:
        feats = [c for c in cols_all if c != t]
        obs = pool[t].notna().to_numpy()
        X, yv = pool.loc[obs, feats], pool.loc[obs, t].to_numpy()
        cut = int(len(X) * 0.9)
        m = lgb.LGBMRegressor(objective="l2", learning_rate=0.06, num_leaves=96,
                              min_child_samples=100, feature_fraction=0.8,
                              bagging_fraction=0.8, bagging_freq=1, n_estimators=600,
                              verbose=-1, n_jobs=-1, seed=seed)
        m.fit(X.iloc[:cut], yv[:cut], eval_set=[(X.iloc[cut:], yv[cut:])],
              callbacks=[lgb.early_stopping(50, verbose=False)])
        pred = m.predict(pool[feats])
        r2 = 1 - ((yv[cut:] - m.predict(X.iloc[cut:])) ** 2).sum() / ((yv[cut:] - yv[cut:].mean()) ** 2).sum()
        filled = pool[t].to_numpy(dtype="float64").copy()
        filled[~obs] = pred[~obs]
        if t == "daily_screen_time_hours":       # respect the generator's hard constraint
            lo = (pool.social_media_hours.fillna(0) + pool.gaming_hours.fillna(0)
                  + pool.work_study_hours.fillna(0)).to_numpy()
            filled[~obs] = np.maximum(filled[~obs], lo[~obs])
        filled = np.round(filled, 2)             # snap back onto the 0.01 lattice
        out[t] = (filled[:n_tr], filled[n_tr:], r2)
    return out

imp = fit_impute(train, test, NUM)
for c, (_, _, r2) in imp.items():
    print(f"  {c:26s} holdout R2 = {r2:.4f}")

## 4. Target encoding — the core of the solution

**Leave-inner-fold-out**, so a training row's encoding never contains its own label.
Validation and test rows use the full outer-fold statistics.

Leakage check performed during development: train-OOF encoding AUC 0.7489 vs honest
holdout 0.7470 on `notifications_per_day`. If it leaked, the training figure would be far
higher.

Two rejected alternatives, both measured: LightGBM **native categorical** on these columns
scored *worse* than raw numeric (0.9602 vs 0.9637) — its one-vs-rest grouping overfits 231
levels. And `max_bin` up to 4096 gained only +0.00003, because finer bins and value
encoding are **substitutes**, not additive.

In [ ]:
NA_CODE = -999_999

def codes_of(s, step):
    v = np.round(s.to_numpy(dtype="float64") / step)
    return np.where(np.isnan(v), NA_CODE, v).astype("int64")


def code_frame(df, cols, cat_cols=()):
    out = {c: codes_of(df[c], STEP[c]) for c in cols}
    for c in cat_cols:
        out[c] = pd.Categorical(df[c]).codes.astype("int64")
    return out


def oof_target_encode(tr_codes, y_tr, other_codes, n_inner=10, smooth=SMOOTH, seed=0):
    # Leave-inner-fold-out TE for training rows; full-fold statistics for everything else.
    # searchsorted, not an offset lookup table: interaction keys span a huge but sparse
    # integer range and indexing by (code - min) would try to allocate that whole range.
    prior = float(y_tr.mean())
    allc = np.concatenate([tr_codes] + list(other_codes))
    uniq = np.unique(allc)
    dense = lambda c: np.searchsorted(uniq, c)
    n = len(uniq)

    ltr = dense(tr_codes)
    tot_cnt = np.bincount(ltr, minlength=n).astype("float64")
    tot_sum = np.bincount(ltr, weights=y_tr.astype("float64"), minlength=n)

    te_tr = np.empty(len(tr_codes))
    for _, fi in StratifiedKFold(n_inner, shuffle=True, random_state=seed).split(np.zeros(len(y_tr)), y_tr):
        f_cnt = np.bincount(ltr[fi], minlength=n).astype("float64")
        f_sum = np.bincount(ltr[fi], weights=y_tr[fi].astype("float64"), minlength=n)
        te_tr[fi] = ((tot_sum - f_sum + prior * smooth) / (tot_cnt - f_cnt + smooth))[ltr[fi]]

    full = (tot_sum + prior * smooth) / (tot_cnt + smooth)
    return te_tr, tot_cnt[ltr], [(full[dense(c)], tot_cnt[dense(c)]) for c in other_codes]

In [ ]:
codes_tr, codes_te = code_frame(train, NUM, CAT), code_frame(test, NUM, CAT)
for c, (a, b, _) in imp.items():
    feat_tr[f"imp_{c}"] = a.astype("float32");  feat_te[f"imp_{c}"] = b.astype("float32")
    feat_tr[f"was_na_{c}"] = train[c].isna().to_numpy().astype("float32")
    feat_te[f"was_na_{c}"] = test[c].isna().to_numpy().astype("float32")
    codes_tr[f"impte_{c}"] = codes_of(pd.Series(a), 0.01)
    codes_te[f"impte_{c}"] = codes_of(pd.Series(b), 0.01)

TE_COLS = NUM + list(CAT) + [f"impte_{c}" for c in imp]
print(f"{len(TE_COLS)} columns target-encoded")


def build_fold(ti, vi):
    ptr, pva, pte = {}, {}, {}
    for c in TE_COLS:
        t_, cn_, oth = oof_target_encode(codes_tr[c][ti], y[ti], [codes_tr[c][vi], codes_te[c]])
        (v_, cv_), (s_, cs_) = oth
        ptr[f"te_{c}"], pva[f"te_{c}"], pte[f"te_{c}"] = t_, v_, s_
        ptr[f"cn_{c}"], pva[f"cn_{c}"], pte[f"cn_{c}"] = cn_, cv_, cs_
    mk = lambda idx, p: pd.concat([train.iloc[idx][NUM].reset_index(drop=True),
                                   feat_tr.iloc[idx].reset_index(drop=True),
                                   pd.DataFrame(p)], axis=1)
    X, V = mk(ti, ptr), mk(vi, pva)
    T = pd.concat([test[NUM].reset_index(drop=True), feat_te.reset_index(drop=True),
                   pd.DataFrame(pte)], axis=1)
    for d in (X, V, T):
        for c in d.columns:
            d[c] = pd.to_numeric(d[c], errors="coerce").replace([np.inf, -np.inf], np.nan).astype("float32")
    return X, V, T

## 5. Model zoo

Five families. LightGBM's pip wheel is CPU-only while XGBoost and CatBoost run on the GPU,
so they are executed on **concurrent streams** — per-fold cost becomes `max(streams)`
rather than their sum (measured 4.1x throughput).

The NNs use **entity embeddings**: one free parameter vector per exact lattice value, which
is the natural representation for a lookup table. Missing values get a dedicated level
rather than being imputed.

> **Why these five.** Adding three more GBDT families was worth +0.00011 to the blend while
> the first NN alone was worth +0.00054 — trees are all wrong in the same places. Judge a
> candidate by how it decorrelates, not by its solo score. HistGradientBoosting and a
> logistic model were both tried and earned exactly zero weight.

In [ ]:
import torch.nn as nn

class EmbMLP(nn.Module):
    def __init__(self, sizes, n_cont, hidden=(256, 256, 256, 128), dropout=0.10,
                 emb_cap=16, act="relu"):
        super().__init__()
        dims = [min(emb_cap, max(4, (s + 1) // 3)) for s in sizes]
        self.embs = nn.ModuleList([nn.Embedding(s, d) for s, d in zip(sizes, dims)])
        self.emb_drop = nn.Dropout(0.05)
        self.bn_cont = nn.BatchNorm1d(n_cont) if n_cont else None
        Act = {"silu": nn.SiLU, "gelu": nn.GELU, "relu": nn.ReLU}[act]
        d_in, layers = sum(dims) + n_cont, []
        for h in hidden:
            layers += [nn.Linear(d_in, h), nn.BatchNorm1d(h), Act(), nn.Dropout(dropout)]
            d_in = h
        self.mlp = nn.Sequential(*layers, nn.Linear(d_in, 1))
        for e in self.embs:
            nn.init.normal_(e.weight, 0, 0.05)

    def forward(self, xc, xn):
        z = self.emb_drop(torch.cat([e(xc[:, i]) for i, e in enumerate(self.embs)], 1))
        if self.bn_cont is not None:
            z = torch.cat([z, self.bn_cont(xn)], 1)
        return self.mlp(z).squeeze(1)


def train_nn(Xc_tr, Xn_tr, y_tr, Xc_va, Xn_va, y_va, Xc_te, Xn_te, sizes,
             epochs=16, bs=2048, lr=4e-3, wd=1e-5, seed=0, arch=None, device=DEVICE):
    torch.manual_seed(seed); np.random.seed(seed)
    model = EmbMLP(sizes, Xn_tr.shape[1], **(arch or {})).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    steps = int(np.ceil(len(y_tr) / bs)) * epochs
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=steps, pct_start=0.25)
    lossf = nn.BCEWithLogitsLoss()
    to = lambda a: torch.from_numpy(a).to(device)          # dataset resident on GPU
    tc, tn, ty = to(Xc_tr), to(Xn_tr), to(y_tr.astype("float32"))
    vc, vn, sc, sn = to(Xc_va), to(Xn_va), to(Xc_te), to(Xn_te)

    @torch.no_grad()
    def predict(c, n_):
        model.eval()
        return torch.sigmoid(torch.cat([model(c[i:i+32768], n_[i:i+32768])
                                        for i in range(0, len(c), 32768)])).float().cpu().numpy()

    best, best_va, best_te, bad = -1, None, None, 0
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(len(ty), device=device)
        for i in range(0, len(ty), bs):
            idx = perm[i:i+bs]
            opt.zero_grad(set_to_none=True)
            lossf(model(tc[idx], tn[idx]), ty[idx]).backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step(); sched.step()
        pv = predict(vc, vn); a = roc_auc_score(y_va, pv)
        if a > best:
            best, bad, best_va, best_te = a, 0, pv, predict(sc, sn)
        else:
            bad += 1
            if bad >= 4:
                break
    return best_va, best_te, best

In [ ]:
# ---- NN inputs: value->id maps use no target, so they are fold-independent ----
def make_levels(tr, te, cols):
    maps, sizes = {}, {}
    for c in cols:
        pool = pd.concat([tr[c], te[c]])
        v = np.round(pool.to_numpy(dtype="float64") / STEP[c])
        u = np.unique(v[~np.isnan(v)])
        maps[c] = {k: i + 1 for i, k in enumerate(u)}   # index 0 reserved for NaN
        sizes[c] = len(u) + 1
    return maps, sizes

maps, sizes_d = make_levels(train, test, NUM)
cat_maps = {c: {v: i + 1 for i, v in enumerate(sorted(set(train[c].dropna()) | set(test[c].dropna())))}
            for c in CAT}

def encode_all(df):
    a = np.zeros((len(df), len(NUM)), dtype="int64")
    for j, c in enumerate(NUM):
        v = np.round(df[c].to_numpy(dtype="float64") / STEP[c])
        a[:, j] = pd.Series(v).map(maps[c]).fillna(0).astype("int64").to_numpy()
    b = np.zeros((len(df), len(CAT)), dtype="int64")
    for j, c in enumerate(CAT):
        b[:, j] = df[c].map(cat_maps[c]).fillna(0).astype("int64").to_numpy()
    return np.hstack([a, b])

Xc_tr, Xc_te = encode_all(train), encode_all(test)
sizes = [sizes_d[c] for c in NUM] + [len(cat_maps[c]) + 1 for c in CAT]

A = pd.concat([train[NUM].reset_index(drop=True), feat_tr.reset_index(drop=True)], axis=1)
B = pd.concat([test[NUM].reset_index(drop=True), feat_te.reset_index(drop=True)], axis=1)
A = A.apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
B = B.apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
mu, sd = A.mean(), A.std().replace(0, 1)
Xn_tr = np.hstack([((A - mu) / sd).fillna(0).to_numpy("float32"), A.isna().to_numpy("float32")])
Xn_te = np.hstack([((B - mu) / sd).fillna(0).to_numpy("float32"), B.isna().to_numpy("float32")])
print("embeddings:", len(sizes), "| continuous:", Xn_tr.shape[1])

In [ ]:
def f_xgb(X, yt, V, yv, T, s):
    m = xgb.XGBClassifier(objective="binary:logistic", eval_metric="auc", tree_method="hist",
                          device=DEVICE, learning_rate=0.04, max_depth=8, min_child_weight=40,
                          subsample=0.8, colsample_bytree=0.55, reg_lambda=5.0, reg_alpha=0.5,
                          n_estimators=5000, early_stopping_rounds=150, n_jobs=4,
                          random_state=s, verbosity=0)
    m.fit(X, yt, eval_set=[(V, yv)], verbose=False)
    return m.predict_proba(V)[:, 1], m.predict_proba(T)[:, 1]


def f_cat(X, yt, V, yv, T, s):
    kw = dict(task_type="GPU", devices="0") if DEVICE == "cuda" else dict(thread_count=4)
    m = CatBoostClassifier(loss_function="Logloss", eval_metric="AUC", learning_rate=0.06,
                           depth=7, l2_leaf_reg=6.0, iterations=4000, random_seed=s,
                           od_type="Iter", od_wait=150, verbose=0,
                           allow_writing_files=False, **kw)
    m.fit(X, yt, eval_set=(V, yv), use_best_model=True)
    return m.predict_proba(V)[:, 1], m.predict_proba(T)[:, 1]


def f_lgb(X, yt, V, yv, T, s):
    m = lgb.LGBMClassifier(objective="binary", metric="auc", learning_rate=0.035, num_leaves=96,
                           min_child_samples=200, feature_fraction=0.55, bagging_fraction=0.8,
                           bagging_freq=1, lambda_l1=0.5, lambda_l2=5.0, n_estimators=6000,
                           verbose=-1, n_jobs=10, seed=s)
    m.fit(X, yt, eval_set=[(V, yv)], callbacks=[lgb.early_stopping(150, verbose=False)])
    return m.predict_proba(V)[:, 1], m.predict_proba(T)[:, 1]


NN_ARCH = {
    "nnC": dict(arch=dict(hidden=(256, 256, 256, 128), dropout=.10, emb_cap=16, act="relu"),
                lr=4e-3, bs=2048, epochs=16),
    "nnB": dict(arch=dict(hidden=(1024, 256), dropout=.25, emb_cap=40, act="gelu"),
                lr=2e-3, bs=8192, epochs=22),
}
MODELS = ["xgb", "cat", "lgb"] + list(NN_ARCH)


def seed_avg(fn, X, yt, V, yv, T, base):
    vs, ts = zip(*[fn(X, yt, V, yv, T, base + 1000 * s) for s in range(N_SEEDS)])
    return np.mean(vs, 0), np.mean(ts, 0)

## 6. Cross-validated training

In [ ]:
oof = {k: np.zeros(len(train)) for k in MODELS}
tep = {k: np.zeros(len(test))  for k in MODELS}
t0 = time.time()

for i, (ti, vi) in enumerate(StratifiedKFold(N_SPLITS, shuffle=True, random_state=SEED).split(train, y)):
    X, V, T = build_fold(ti, vi)
    print(f"\n--- fold {i+1}/{N_SPLITS}  ({X.shape[1]} features, {time.time()-t0:.0f}s) ---", flush=True)
    res = {}
    with ThreadPoolExecutor(max_workers=1) as ex:
        fut = ex.submit(seed_avg, f_lgb, X, y[ti], V, y[vi], T, SEED + i)   # CPU stream
        res["xgb"] = seed_avg(f_xgb, X, y[ti], V, y[vi], T, SEED + i)       # GPU stream
        res["cat"] = seed_avg(f_cat, X, y[ti], V, y[vi], T, SEED + i)
        for nm, cfg in NN_ARCH.items():
            vs, ts = [], []
            for s in range(N_SEEDS):
                vp, tp, _ = train_nn(Xc_tr[ti], Xn_tr[ti], y[ti], Xc_tr[vi], Xn_tr[vi], y[vi],
                                     Xc_te, Xn_te, sizes, seed=SEED + i + 1000 * s,
                                     arch=cfg["arch"], lr=cfg["lr"], bs=cfg["bs"], epochs=cfg["epochs"])
                vs.append(vp); ts.append(tp)
            res[nm] = (np.mean(vs, 0), np.mean(ts, 0))
        res["lgb"] = fut.result()

    for k in MODELS:
        vp, tp = res[k]
        oof[k][vi] = vp; tep[k] += tp / N_SPLITS
        print(f"    {k:4s} AUC={roc_auc_score(y[vi], vp):.6f}")
    for k in MODELS:                       # checkpoint -- long runs must survive a sleep
        np.save(f"{OUT}/oof_{k}.npy", oof[k])
        np.save(f"{OUT}/test_{k}.npy", tep[k] * N_SPLITS / (i + 1))

print("\n===== per-model OOF =====")
scores = {k: roc_auc_score(y, oof[k]) for k in MODELS}
for k, v in sorted(scores.items(), key=lambda x: -x[1]):
    print(f"  {k:5s} {v:.6f}")

## 7. Blending

**Rank space, not probability space** — the families put their outputs on different scales
and AUC only cares about ordering, so averaging ranks removes a nuisance degree of freedom.

Greedy forward selection with replacement (Caruana), seeded with several copies of the best
model so the first increment is fine-grained. Weight-fitting optimism was measured at
**+0.000008** by fitting on half the OOF and scoring the other half — negligible.

> **Known improvement, not used in the submitted solution.** The 7th-place team extracted
> **+0.0018** from blending against our +0.00061, using six cross-fitted logistic regressions
> on rank-normalised predictions and — critically — **allowing negative weights**. Greedy
> selection is non-negative by construction and cannot express "this direction is
> over-represented". See `kaggle_learnings.md`.

In [ ]:
R = {k: rankdata(oof[k]) / len(oof[k]) for k in MODELS}
Rt = {k: rankdata(tep[k]) / len(tep[k]) for k in MODELS}

INIT_K, N_ITER = 5, 60
best0 = max(MODELS, key=lambda k: scores[k])
cur = R[best0].copy()
counts = {k: 0 for k in MODELS}; counts[best0] = INIT_K; n = INIT_K
best_overall, best_counts = scores[best0], dict(counts)
print(f"init: {INIT_K}x {best0}  OOF = {best_overall:.6f}")

for it in range(N_ITER):
    cand = {k: roc_auc_score(y, (cur * n + R[k]) / (n + 1)) for k in MODELS}
    bk = max(cand, key=cand.get)
    cur = (cur * n + R[bk]) / (n + 1); n += 1; counts[bk] += 1
    if cand[bk] > best_overall + 1e-9:
        best_overall, best_counts = cand[bk], dict(counts)
        print(f"  iter {it+1:2d}: +{bk:5s} OOF = {cand[bk]:.6f}  *")

tot = sum(best_counts.values())
weights = {k: v / tot for k, v in best_counts.items() if v}
blend_oof = sum(w * R[k] for k, w in weights.items())
blend_te  = sum(w * Rt[k] for k, w in weights.items())

print("\n=== final blend ===")
for k, w in sorted(weights.items(), key=lambda x: -x[1]):
    print(f"  {k:5s} {w:.3f}")
print(f"  blend OOF   = {roc_auc_score(y, blend_oof):.6f}")
print(f"  best single = {scores[best0]:.6f} ({best0})")
print(f"  gain        = {roc_auc_score(y, blend_oof) - scores[best0]:+.6f}")

## 8. Submission

**Calibration.** The CV↔LB offset was measured by nested holdout, not assumed: run the
K-fold pipeline on 80% and score the *averaged* fold-models on an untouched 20%. The
bagging offset came out at **−0.000016** — the ensembling gain (+0.00048) is cancelled by
OOF optimism (+0.00050), because each fold-model early-stops on the fold it is then scored
on. So `LB ≈ OOF`, and the observed offset on the real submission was +0.00103, consistent
with the public split being an easy draw.

In [ ]:
submission = pd.DataFrame({"id": test["id"].values, TARGET: blend_te})
submission.to_csv(f"{OUT}/submission.csv", index=False)

sample = pd.read_csv(f"{DATA}/sample_submission.csv")
assert list(submission.columns) == list(sample.columns)
assert len(submission) == len(sample)
assert (submission.id.values == sample.id.values).all()
assert submission[TARGET].notna().all() and np.isfinite(submission[TARGET]).all()
print(f"submission.csv OK -- {len(submission):,} rows, "
      f"range [{submission[TARGET].min():.6f}, {submission[TARGET].max():.6f}]")
submission.head()

---

## What did not work (all measured)

| idea | result |
|---|---|
| Hard-coding the source rule (`daily>8 or social>4`) | 0.865 — the generator smeared it away |
| LightGBM native categorical on lattice columns | 0.9602, *worse* than raw numeric |
| Numeric missingness flags / `n_na` | ~0 — missingness is MCAR (AUC 0.502 alone) |
| Training on the original 7,500-row dataset | every fold worse — different label mechanism |
| 2D binned target encoding | 0.967740 vs 0.967744 — GBDT finds it unaided |
| `max_bin` 256 → 4096 | +0.00003, zero blend weight |
| Periodicity (sin/cos) features | **−0.00011** |
| Pseudo-labelling (4 variants) | +0.00006 at best |
| Seed averaging on top of blending | +0.000004 — the blend already averages that variance |
| HistGradientBoosting, logistic on TE | zero blend weight |

**The unifying explanation:** `max_bin`, decimal features, frequency encoding and
periodicity are all mechanisms for *separating individual values*. Exact-value target
encoding already does that completely, so each measures large against a raw-feature
baseline and about zero here. Always ask what baseline a quoted number came from.

## Post-mortem

Finished **717 / 3532**. Against 7th place's published OOF table our individual models were
competitive — XGBoost/LightGBM/CatBoost within 0.0003, and our hand-rolled MLP *better*
than theirs (0.968106 vs 0.967608). The gap was **ensembling**: they turned 0.969080 into
0.970879 (**+0.0018**); we turned 0.968729 into 0.969340 (**+0.00061**).

They used ~20 model families to our 5, 556 prediction streams to our 14, and a meta-model
that permitted negative weights. Full analysis in `kaggle_learnings.md`.